# Поиск нетипичного поведения заявок на закупку

Анализ журнала событий `students.csv` и формирование `anomalies.csv`.

# Загрузка данных и структура


In [1]:
import os
import pandas as pd
import numpy as np
from datetime import timedelta

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 200)

DATA_DIR = 'Case1'
OUT_PATH = os.path.join('ResultCase1', 'anomalies.csv')
SNAPSHOT = pd.Timestamp('2026-07-01T00:00:00+03:00')  # срез наблюдения

def resolve_path(name):
    '''Гибкий поиск файла данных (на случай запуска из разных каталогов).'''
    candidates = [
        os.path.join('Case1', name),
        name,
        os.path.join('..', 'Case1', name),
        os.path.join('..', 'Elma1', name),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0]

CSV_PATH = resolve_path('students.csv')

# Явная кодировка этапов (НЕ зависит от порядка появления в файле)
ACTIVITY_CODE = {
    'Регистрация заявки':           '0',
    'Проверка заявки':              '1',
    'Согласование руководителем':   '2',
    'Завершение':                   '3',
    'Согласование бюджета':         '4',
    'Доработка заявки':             '5',
    'Исполнение заявки':            '6',
    'Проверка договора':            '7',
    'Внеплановая проверка':         '8',
}
CODE_NAME = {v: k for k, v in ACTIVITY_CODE.items()}

# Ожидаемая роль на каждом человеческом этапе
EXPECTED_ROLE = {
    'Проверка заявки':              'specialist',
    'Согласование руководителем':   'manager',
    'Согласование бюджета':         'finance',
    'Доработка заявки':             'requester',
    'Исполнение заявки':            'procurement',
    'Проверка договора':            'legal',
    'Внеплановая проверка':         'auditor',
}

# Обязательные этапы (в правильном порядке, без завершения '3')
MANDATORY = {
    'standard': ['0', '1', '2', '4', '6'],
    'urgent':   ['0', '1', '2', '4', '6'],
    'contract': ['0', '1', '2', '4', '7', '6'],
}

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

print('=== Структура ===')
print('Строк:', len(df), '| процессов:', df['ProcessInstanceId'].nunique())
print('Колонки:', list(df.columns))

print()
print('=== Качество данных ===')
print('Пустых значений по колонкам:')
print(df.isna().sum().to_string())
print()
print('EventType:', df['EventType'].value_counts(dropna=False).to_dict())
print()
print('RequestType:', df['RequestType'].value_counts(dropna=False).to_dict())
print()
print('Role:', df['Role'].fillna('<пусто>').value_counts().to_dict())
print()
print('Result:', df['Result'].fillna('<пусто>').value_counts().to_dict())
print()
print('ActivityName:')
print(df['ActivityName'].value_counts(dropna=False).to_string())

# Дубликаты EventId / повторное закрытие одного задания
print()
print('Дубликаты EventId:', int(df['EventId'].duplicated().sum()))
dup_closed = df[df['EventType'] == 'task_completed']['TaskId'].duplicated().sum()
print('TaskId с более чем одним закрытием:', int(dup_closed))

# Этапы, которых нет в нашем словаре
print()
unmapped = sorted(set(df['ActivityName'].dropna()) - set(ACTIVITY_CODE))
print('Этапы без кода (если есть — поправьте ACTIVITY_CODE):', unmapped)


=== Структура ===
Строк: 12162 | процессов: 1000
Колонки: ['ProcessInstanceId', 'EventId', 'EventIndex', 'EventType', 'ActivityId', 'ActivityName', 'TaskId', 'Timestamp', 'UserId', 'Role', 'Status', 'Result', 'RequestType', 'Amount', 'TemplateVersion']

=== Качество данных ===
Пустых значений по колонкам:
ProcessInstanceId       0
EventId                 0
EventIndex              0
EventType               0
ActivityId              0
ActivityName            0
TaskId               2990
Timestamp               0
UserId               6581
Role                 5581
Status                  0
Result               6591
RequestType             0
Amount                  0
TemplateVersion         0

EventType: {'task_created': 4591, 'task_completed': 4581, 'process_started': 1000, 'automatic_completed': 1000, 'process_completed': 990}

RequestType: {'standard': 7575, 'contract': 2629, 'urgent': 1958}

Role: {'<пусто>': 5581, 'requester': 1181, 'specialist': 1181, 'manager': 1141, 'system': 1000, 

# Подготовка данных и расчёт рабочего времени


In [2]:
df = df.drop(columns=['TemplateVersion'], errors='ignore')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], utc=True)
df['EventIndex'] = pd.to_numeric(df['EventIndex'], errors='coerce').astype('Int64')
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
df['_code'] = df['ActivityName'].map(ACTIVITY_CODE)

# Строго сортируем события внутри процесса по EventIndex (требование описания данных)
df = df.sort_values(['ProcessInstanceId', 'EventIndex'], kind='mergesort').reset_index(drop=True)


def to_local_naive(series):
    '''Местное время (+03:00) без таймзоны — для расчёта рабочего времени.'''
    return series.dt.tz_convert('+03:00').dt.tz_localize(None)


def _py(x):
    if pd.isna(x):
        return None
    return x.to_pydatetime() if hasattr(x, 'to_pydatetime') else x


def business_hours(start, end):
    '''Рабочие часы между двумя моментами (Пн–Пт 09:00–18:00, без праздников/обеда).'''
    s, e = _py(start), _py(end)
    if s is None or e is None or e <= s:
        return 0.0 if (s is not None and e is not None) else np.nan

    total, cur = 0.0, s

    # выйти в ближайший рабочий интервал
    while True:
        wd = cur.weekday()  # 0=Пн … 6=Вс
        if wd >= 5:
            cur = (cur + timedelta(days=7 - wd)).replace(hour=9, minute=0, second=0, microsecond=0)
            continue
        ds = cur.replace(hour=9, minute=0, second=0, microsecond=0)
        de = cur.replace(hour=18, minute=0, second=0, microsecond=0)
        if cur >= de:
            cur = (cur + timedelta(days=1)).replace(hour=9, minute=0, second=0, microsecond=0)
            continue
        if cur < ds:
            cur = ds
        break

    while cur < e:
        wd = cur.weekday()
        if wd >= 5:
            cur = (cur + timedelta(days=7 - wd)).replace(hour=9, minute=0, second=0, microsecond=0)
            continue
        ds = cur.replace(hour=9, minute=0, second=0, microsecond=0)
        de = cur.replace(hour=18, minute=0, second=0, microsecond=0)
        seg_end = min(e, de)
        total += (seg_end - cur).total_seconds() / 3600.0
        if seg_end >= e:
            break
        cur = (cur + timedelta(days=1)).replace(hour=9, minute=0, second=0, microsecond=0)

    return total


# Сводки по процессам и заданиям (норма)


In [3]:
proc_rows = []
for pid, g in df.groupby('ProcessInstanceId', sort=False):
    started = g.loc[g['EventType'] == 'process_started', 'Timestamp']
    completed = g.loc[g['EventType'] == 'process_completed', 'Timestamp']
    final = g.loc[g['EventType'] == 'process_completed', 'Result']

    start_ts = started.iloc[0] if len(started) else g['Timestamp'].iloc[0]
    end_ts = completed.iloc[0] if len(completed) else pd.NaT
    final_result = final.iloc[0] if len(final) else None

    # цепочка этапов (схлопывание подряд идущих одинаковых кодов)
    codes, prev = [], None
    for c in g['_code'].tolist():
        c = c if pd.notna(c) else '?'
        if c != prev:
            codes.append(c)
            prev = c
    chain = ''.join(codes)

    tc = g[g['EventType'] == 'task_completed']
    proc_rows.append({
        'ProcessInstanceId': pid,
        'RequestType': g['RequestType'].iloc[0],
        'Amount': g['Amount'].iloc[0],
        'amount_nunique': g['Amount'].nunique(),
        'type_nunique': g['RequestType'].nunique(),
        'start_ts': start_ts,
        'end_ts': end_ts,
        'final_result': final_result,
        'chain': chain,
        'num_returns': int((tc['Result'] == 'return').sum()),
        'num_rejects': int((tc['Result'] == 'reject').sum()),
        'is_open': len(completed) == 0,
    })

procs = pd.DataFrame(proc_rows)

procs['duration_cal_h'] = (procs['end_ts'] - procs['start_ts']).dt.total_seconds() / 3600.0
procs['duration_work_h'] = np.nan
closed_mask = ~procs['is_open']
procs.loc[closed_mask, 'duration_work_h'] = [
    business_hours(a, b)
    for a, b in zip(
        to_local_naive(procs.loc[closed_mask, 'start_ts']),
        to_local_naive(procs.loc[closed_mask, 'end_ts']),
    )
]

# Длительность человеческих заданий
created = (
    df[df['EventType'] == 'task_created']
    [['TaskId', 'ProcessInstanceId', 'ActivityName', 'RequestType', 'Timestamp']]
    .rename(columns={'Timestamp': 'created_ts'})
)
closed = (
    df[df['EventType'] == 'task_completed']
    [['TaskId', 'Timestamp', 'Role', 'Result']]
    .rename(columns={'Timestamp': 'closed_ts'})
)

tasks = created.merge(closed, on='TaskId', how='left')
tasks['created_local'] = to_local_naive(tasks['created_ts'])
tasks['closed_local'] = to_local_naive(tasks['closed_ts'])

tasks['dur_cal_h'] = (tasks['closed_ts'] - tasks['created_ts']).dt.total_seconds() / 3600.0
tasks['dur_work_h'] = np.nan
has_close = tasks['closed_ts'].notna()
tasks.loc[has_close, 'dur_work_h'] = [
    business_hours(a, b)
    for a, b in zip(tasks.loc[has_close, 'created_local'], tasks.loc[has_close, 'closed_local'])
]

# возраст незакрытого задания до среза (нижняя граница длительности)
open_task = ~has_close
tasks['age_to_snapshot_h'] = np.nan
tasks.loc[open_task, 'age_to_snapshot_h'] = (
    SNAPSHOT - tasks.loc[open_task, 'created_ts']
).dt.total_seconds() / 3600.0

# Печать «нормы»
print('=== Процессы ===')
print('Всего:', len(procs), '| закрыто:', int((~procs['is_open']).sum()),
      '| открыто на срезе:', int(procs['is_open'].sum()))
print()
print('Завершение по типу:')
print(procs.groupby('RequestType')['final_result'].value_counts(dropna=False).unstack(fill_value=0).to_string())
print()
print('Возвраты (num_returns) по процессам:')
print(procs['num_returns'].value_counts().sort_index().to_string())

print()
print('=== Длительность заданий (рабочие часы), по этапу и типу ===')
task_stat = (
    tasks[tasks['dur_work_h'].notna()]
    .groupby(['ActivityName', 'RequestType'])['dur_work_h']
    .describe(percentiles=[0.5, 0.9, 0.99])
)
print(task_stat[['count', 'mean', '50%', '90%', '99%', 'max']].round(2).to_string())

print()
print('=== Длительность процессов (рабочие часы), закрытые ===')
proc_stat = (
    procs[procs['duration_work_h'].notna()]
    .groupby('RequestType')['duration_work_h']
    .describe(percentiles=[0.5, 0.9, 0.99])
)
print(proc_stat[['count', 'mean', '50%', '90%', '99%', 'max']].round(2).to_string())

print()
print('=== Суммы (Amount) по типу ===')
print(procs.groupby('RequestType')['Amount'].describe(percentiles=[0.25, 0.5, 0.75]).round(2).to_string())


=== Процессы ===
Всего: 1000 | закрыто: 990 | открыто на срезе: 10

Завершение по типу:
final_result  fulfilled  rejected  NaN
RequestType                           
contract            182        14    1
standard            597        30    8
urgent              160         7    1

Возвраты (num_returns) по процессам:
num_returns
0    873
1    119
5      1
6      3
7      1
8      3

=== Длительность заданий (рабочие часы), по этапу и типу ===
                                        count  mean   50%   90%    99%    max
ActivityName               RequestType                                       
Внеплановая проверка       contract       4.0  3.52  3.50  4.99   5.26   5.29
                           standard      10.0  2.45  2.30  3.56   3.92   3.96
                           urgent         2.0  1.46  1.46  1.47   1.47   1.47
Доработка заявки           contract      26.0  1.51  1.40  2.43   3.05   3.10
                           standard     123.0  1.50  1.42  2.41   3.15   3.20
     

# Пороги аномальности


In [4]:
# Порог длительности задания: 99-й перцентиль внутри (этап, тип)
tasks['dur_p99'] = tasks.groupby(['ActivityName', 'RequestType'])['dur_work_h'].transform(
    lambda s: s.quantile(0.99)
)
MIN_TASK_H = 8.0  # один рабочий день: не флагаем тривиальные отличия

# Порог длительности процесса: 99-й перцентиль внутри типа
procs['proc_p99'] = procs.groupby('RequestType')['duration_work_h'].transform(
    lambda s: s.quantile(0.99)
)

# Порог суммы: 99.5-й перцентиль внутри типа (жёсткий порог для «необычно крупных»)
procs['amount_p995'] = procs.groupby('RequestType')['Amount'].transform(lambda s: s.quantile(0.995))

# Порог «избыточных возвратов»
RETURN_LIMIT = max(3, int(procs['num_returns'].quantile(0.99)) + 1)
print('RETURN_LIMIT =', RETURN_LIMIT)


RETURN_LIMIT = 3


# Детекторы аномалий


In [5]:
rows = []  # ProcessInstanceId, AnomalyType, Explanation

def add(pid, atype, expl):
    rows.append({'ProcessInstanceId': pid, 'AnomalyType': atype, 'Explanation': expl})


def names(codes):
    return ', '.join(f'«{CODE_NAME.get(c, c)}»' for c in codes)


# 1. Пропущен обязательный этап
def missing_stage(chain, req_type, result):
    if result == 'fulfilled':
        required = MANDATORY.get(req_type, MANDATORY['standard'])
    elif result == 'rejected':
        required = ['0', '1', '2']
    elif result is None:  # открытый процесс — проверяем по самому дальнему этапу
        order = MANDATORY.get(req_type, MANDATORY['standard'])
        present = [c for c in order if c in chain]
        if not present:
            return None
        required = order[: max(order.index(c) for c in present) + 1]
    else:
        return None
    miss = [c for c in required if c not in chain]
    return miss if miss else None

for _, p in procs.iterrows():
    miss = missing_stage(p['chain'], p['RequestType'], p['final_result'])
    if miss:
        add(p['ProcessInstanceId'], 'Пропущен обязательный этап',
            f"Маршрут {p['chain']} ({p['RequestType']}); не хватает: {names(miss)}.")

# 2. Работа после отказа
for _, p in procs.iterrows():
    if p['final_result'] == 'rejected':
        extra = [c for c in ['4', '6', '7'] if c in p['chain']]
        if extra:
            add(p['ProcessInstanceId'], 'Работа после отказа',
                f"Процесс завершён отказом, но выполнены этапы {names(extra)} (маршрут {p['chain']}).")

# 3. Юрпроверка у недоговорной заявки
for _, p in procs.iterrows():
    if p['RequestType'] != 'contract' and '7' in p['chain']:
        add(p['ProcessInstanceId'], 'Юрпроверка недоговорной заявки',
            f"Этап {names(['7'])} выполнен для заявки типа {p['RequestType']} (юрпроверка обязательна только для contract).")

# 4. Повтор этапа (бюджет/юрист/исполнение)
for _, p in procs.iterrows():
    for c in ['4', '6', '7']:
        n = p['chain'].count(c)
        if n > 1:
            add(p['ProcessInstanceId'], 'Повтор этапа',
                f"Этап {names([c])} повторён {n} раз (маршрут {p['chain']}).")

# 5. Несогласованность цикла возврата
# При отсутствии возвратов не запускаем: это дублирует детектор «пропущен этап».
for _, p in procs.iterrows():
    c1, c2 = p['chain'].count('1'), p['chain'].count('2')
    expected = p['num_returns'] + 1
    if p['num_returns'] > 0 and (c1 != expected or c2 != expected):
        add(p['ProcessInstanceId'], 'Несогласованность цикла возврата',
            f"Проверок={c1}, согласований={c2}, возвратов={p['num_returns']} (ожидалось по {expected}). Маршрут: {p['chain']}.")

# 6. Некорректная роль на этапе
completed = df[df['EventType'] == 'task_completed'].copy()
completed['expected_role'] = completed['ActivityName'].map(EXPECTED_ROLE)
role_bad = completed[
    completed['expected_role'].notna() & (completed['Role'] != completed['expected_role'])
]
for (pid, aname), grp in role_bad.groupby(['ProcessInstanceId', 'ActivityName']):
    actual = ', '.join(sorted(set(grp['Role'].fillna('<пусто>').astype(str))))
    add(pid, 'Некорректная роль',
        f"На этапе «{aname}» роль «{actual}» вместо ожидаемой «{EXPECTED_ROLE[aname]}» (событий: {len(grp)}).")

# 7. Аномальная длительность задания
out = tasks[tasks['dur_work_h'].notna() & (tasks['dur_work_h'] > tasks['dur_p99'] * 1.5) & (tasks['dur_work_h'] >= MIN_TASK_H)]
for _, t in out.iterrows():
    add(t['ProcessInstanceId'], 'Аномальная длительность этапа',
        f"«{t['ActivityName']}» длился {t['dur_work_h']:.1f} раб.ч (99-й перцентиль {t['RequestType']}: {t['dur_p99']:.1f} раб.ч).")

# 8. Аномальная длительность процесса
out_p = procs[procs['duration_work_h'].notna() & (procs['duration_work_h'] > procs['proc_p99'])]
for _, p in out_p.iterrows():
    add(p['ProcessInstanceId'], 'Аномальная длительность процесса',
        f"Процесс {p['RequestType']} длился {p['duration_work_h']:.1f} раб.ч ({p['duration_cal_h']/24:.1f} кал.дней); 99-й перцентиль типа — {p['proc_p99']:.1f} раб.ч.")

# 9. Необычно крупная сумма (не нарушение, но для проверки)
out_a = procs[procs['Amount'].notna() & (procs['Amount'] > procs['amount_p995'])]
for _, p in out_a.iterrows():
    add(p['ProcessInstanceId'], 'Необычно крупная сумма',
        f"Сумма {p['Amount']:.2f} усл.руб. выше 99.5-го перцентиля ({p['amount_p995']:.2f}) для типа {p['RequestType']} (необычно, но допустимо).")

# 10. Избыточные возвраты
for _, p in procs.iterrows():
    if p['num_returns'] >= RETURN_LIMIT:
        add(p['ProcessInstanceId'], 'Избыточные возвраты',
            f"{p['num_returns']} возвратов на доработку (порог {RETURN_LIMIT}); у большинства заявок ≤1 возврат. Маршрут: {p['chain']}.")

# 11. Сумма/тип менялись внутри процесса (качество данных)
for _, p in procs.iterrows():
    if p['amount_nunique'] > 1:
        add(p['ProcessInstanceId'], 'Сумма изменялась внутри процесса',
            f"У процесса {p['amount_nunique']} различных значений Amount (ожидается одно).")
    if p['type_nunique'] > 1:
        add(p['ProcessInstanceId'], 'Тип заявки изменялся внутри процесса',
            f"У процесса {p['type_nunique']} различных RequestType (ожидается один).")

# 12. Повторное закрытие одного задания (качество данных)
dup_closed_ids = (
    df[df['EventType'] == 'task_completed']
    .groupby('TaskId')['EventId'].nunique()
)
for tid in dup_closed_ids[dup_closed_ids > 1].index:
    pid = df.loc[df['TaskId'] == tid, 'ProcessInstanceId'].iloc[0]
    add(pid, 'Дублирование закрытия задания',
        f'Задание {tid} закрыто более одного раза.')


# Формирование anomalies.csv


In [6]:
anomalies = pd.DataFrame(rows, columns=['ProcessInstanceId', 'AnomalyType', 'Explanation'])
anomalies = anomalies.drop_duplicates().sort_values(['ProcessInstanceId', 'AnomalyType']).reset_index(drop=True)

print('=== Итог ===')
print('Строк-аномалий:', len(anomalies))
print('Уникальных процессов с аномалиями:', anomalies['ProcessInstanceId'].nunique())
print()
print('По типам:')
print(anomalies['AnomalyType'].value_counts().to_string())
print()
print('Примеры:')
print(anomalies.head(20).to_string(index=False))

anomalies.to_csv(OUT_PATH, index=False, encoding='utf-8')
print()
print('Записано:', OUT_PATH)

# Справка для отчёта: внеплановые проверки и открытые процессы
print()
print('=== Для ручной проверки ===')
print('Процессы с внеплановой проверкой (auditor):')
print(procs[procs['chain'].str.contains('8', na=False)]['ProcessInstanceId'].to_string(index=False))
open_procs = procs[procs['is_open']]
print()
print('Открытых процессов на срезе:', len(open_procs), '— сами по себе не аномалия.')
if len(open_procs):
    print('Самые старые открытые (по старту):')
    print(open_procs.sort_values('start_ts')[['ProcessInstanceId', 'RequestType', 'start_ts', 'chain']].head(10).to_string(index=False))


=== Итог ===
Строк-аномалий: 57
Уникальных процессов с аномалиями: 38

По типам:
AnomalyType
Пропущен обязательный этап          16
Аномальная длительность процесса    11
Аномальная длительность этапа        8
Некорректная роль                    8
Избыточные возвраты                  8
Необычно крупная сумма               6

Примеры:
                   ProcessInstanceId                      AnomalyType                                                                                                   Explanation
10daf811-5f75-5dbf-a76c-b4f818fc2e2d       Пропущен обязательный этап                                           Маршрут 01463 (standard); не хватает: «Согласование руководителем».
17b4a994-a650-5539-979c-114b14dc20d6 Аномальная длительность процесса                          Процесс urgent длился 63.3 раб.ч (11.0 кал.дней); 99-й перцентиль типа — 28.0 раб.ч.
17b4a994-a650-5539-979c-114b14dc20d6    Аномальная длительность этапа                                      «Проверка заявки

## Ограничения метода

- Длительность — календарное время, включая ожидание; рабочее время считается
  по календарю Пн–Пт 09:00–18:00 без праздников и обедов.
- Открытые процессы дают только нижнюю границу длительности; сами по себе они
  не аномалия (см. раздел «ручная проверка»).
- Пороги — 99-й перцентиль / IQR — зависят от выборки; пограничные случаи
  требуют ручной проверки.
- Внеплановая проверка (auditor) не считается аномалией только по редкости.

## Структура отчёта (по submission_template.md)

1. Инструменты и воспроизводимые шаги (этот ноутбук).
2. Что считаем нормой (выводы ячеек с метриками).
3. Критерии поиска и пороги.
4. Результаты (количество кандидатов и примеры).
5. Ложные срабатывания, ограничения, открытые процессы, ручная проверка.
6. Использование AI (роль и как проверялись ответы).
